# 05 — Mitigation: GRU + Adversarial Debiasing (In-processing)

This notebook implements **adversarial debiasing** as an in-processing fairness mitigation strategy applied to a GRU-based toxicity classifier trained on the Jigsaw dataset.

| Item | Detail |
|------|--------|
| **Model** | Bidirectional GRU with Gradient-Reversal Layer (GRL) |
| **Tokenizer** | BERT WordPiece (`bert-base-uncased`) |
| **Mitigation** | Adversarial debiasing via Gradient Reversal Layer |
| **Fairness metrics** | Subgroup AUC, BPSN AUC, BNSP AUC, FPR by group, ECE |
| **Counterfactual** | Swap-word gap (black/white, christian/muslim, male/female) |

## 0. Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, math, numpy as np, pandas as pd, torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.autograd import Function
from sklearn.metrics import classification_report, roc_auc_score
from transformers import BertTokenizer
from fairness_jigsaw.metrics import DEFAULT_IDENTITY_COLUMNS, ModelBiasEvaluator

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {DEVICE}")
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)
TOXICITY_THRESHOLD = 0.5; IDENTITY_THRESHOLD = 0.5
N_TRAIN_SAMPLE = None
MAX_LEN = 128
EMBED_DIM = 64; HIDDEN_DIM = 128
BATCH_SIZE = 256; EPOCHS = 3; LR = 1e-3
LAMBDA_ADV = 0.5
ADV_ALPHA  = 2.0   # reduced from 10.0: sigmoid ramp is near-step with 3 epochs at 10.0

Device: mps


## 1. Data

We reuse the pre-computed `split_ids.json` to obtain identical train / validation / test splits as the baseline notebooks.  
The dataset is the [Jigsaw Unintended Bias in Toxicity Classification](https://www.kaggle.com/c/jigsaw-unintended-bias-in-toxicity-classification) dataset.  
Each row has a `target` score (continuous 0–1) which we binarise at `TOXICITY_THRESHOLD = 0.5`.

In [2]:
with open("../data/split_ids.json") as f:
    split_ids = json.load(f)
USE_COLS = ["id", "target", "comment_text"] + list(DEFAULT_IDENTITY_COLUMNS)
df_all = pd.read_csv("../data/train.csv", usecols=USE_COLS)
df_all["toxic"] = (df_all["target"] >= TOXICITY_THRESHOLD).astype(int)
train_df = df_all[df_all["id"].isin(split_ids["train"])].reset_index(drop=True)
val_df   = df_all[df_all["id"].isin(split_ids["val"])].reset_index(drop=True)
test_df  = df_all[df_all["id"].isin(split_ids["test"])].reset_index(drop=True)
if N_TRAIN_SAMPLE is not None:
    train_df = train_df.sample(n=N_TRAIN_SAMPLE, random_state=SEED).reset_index(drop=True)
for name, df in [("Train", train_df), ("Val  ", val_df), ("Test ", test_df)]:
    anno = df[list(DEFAULT_IDENTITY_COLUMNS)].notna().any(axis=1).sum()
    print(f"{name}: {len(df):>10,}  | toxic: {df['toxic'].mean():.2%}  | annotated: {anno:>7,} ({anno/len(df):.1%})")

Train:  1,443,897  | toxic: 8.00%  | annotated: 324,097 (22.4%)
Val  :    180,486  | toxic: 8.00%  | annotated:  40,486 (22.4%)
Test :    180,491  | toxic: 8.00%  | annotated:  40,547 (22.5%)


## 2. Text Preprocessing

Load a **BERT WordPiece** tokenizer (`bert-base-uncased`). Sequences are truncated or padded to `MAX_LEN`.

In [3]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print(f"BERT vocabulary size : {tokenizer.vocab_size:,} tokens")
print(f"Example tokens       : {tokenizer.tokenize('I hate this person')}")


def encode(text) -> list[int]:
    return tokenizer.encode(
        text if isinstance(text, str) else "",
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )

BERT vocabulary size : 30,522 tokens
Example tokens       : ['i', 'hate', 'this', 'person']


## 3. Adversarial Model

The **Gradient Reversal Layer (GRL)** is the core of adversarial debiasing.  
During the **forward pass**, the GRL acts as an identity function — the hidden representation `rep` passes through unchanged to both the toxicity classifier and the adversary head.  
During the **backward pass**, gradients flowing from the adversary are **negated** (multiplied by `-alpha`) before reaching the GRU encoder, actively discouraging the encoder from learning identity-predictive representations.  
As a result, the shared encoder is penalised for retaining demographic signals, pushing it towards a fairer, identity-invariant feature space.

- `LAMBDA_ADV` controls the **overall strength** of the adversarial penalty relative to the classification loss.  
- `ADV_ALPHA` controls the **ramp speed** of the reversal strength across epochs — a sigmoid schedule ensures a smooth warm-up rather than an abrupt gradient reversal from epoch one.

In [4]:
class GRL(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class FairGRU(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, n_identities: int) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.adversary = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_identities),
        )

    def forward(self, x: torch.Tensor, alpha: float = 0.0):
        emb = self.embedding(x)
        _, h = self.gru(emb)
        rep = h[-1]
        tox = torch.sigmoid(self.classifier(rep)).squeeze(-1)
        adv = torch.sigmoid(self.adversary(GRL.apply(rep, alpha)))
        return tox, adv

model = FairGRU(tokenizer.vocab_size, EMBED_DIM, HIDDEN_DIM, len(DEFAULT_IDENTITY_COLUMNS)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {n_params:,}")

FairGRU(
  (embedding): Embedding(30522, 64, padding_idx=0)
  (gru): GRU(64, 128, batch_first=True)
  (classifier): Linear(in_features=128, out_features=1, bias=True)
  (adversary): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=24, bias=True)
  )
)

Total parameters: 2,037,849


## 4. Dataset & Training

`AdvCommentDataset` extends the standard comment dataset with **binarised identity labels** (one per identity column) used as adversary targets.  
`CommentDataset` is the lightweight version used for validation and test inference (no identity labels needed).

The training loop uses a **sigmoid ramp schedule** for the GRL reversal strength `alpha`:

$$\alpha_t = \lambda_{\text{adv}} \cdot \left(\frac{2}{1 + e^{-\alpha_0 \cdot \text{progress}}} - 1\right)$$

where `progress` runs from 0 to 1 over training epochs.  
At epoch 1, `alpha ≈ 0` (minimal reversal); by the final epoch the adversary is fully engaged.

In [5]:
# Binarise identity columns (needed for adversary labels)
for col in DEFAULT_IDENTITY_COLUMNS:
    train_df[f"{col}_bin"] = (train_df[col].fillna(0) >= IDENTITY_THRESHOLD).astype(int)

class AdvCommentDataset(Dataset):
    def __init__(self, df: pd.DataFrame) -> None:
        self.x = torch.tensor([encode(t) for t in df["comment_text"]], dtype=torch.long)
        self.y_tox = torch.tensor(df["toxic"].values, dtype=torch.float32)
        bin_cols = [f"{c}_bin" for c in DEFAULT_IDENTITY_COLUMNS]
        self.y_id = torch.tensor(df[bin_cols].fillna(0).values, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y_tox[idx], self.y_id[idx]

class CommentDataset(Dataset):
    def __init__(self, df: pd.DataFrame) -> None:
        self.x = torch.tensor([encode(t) for t in df["comment_text"]], dtype=torch.long)
        self.y = torch.tensor(df["toxic"].values, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_loader = DataLoader(AdvCommentDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(CommentDataset(test_df),  batch_size=BATCH_SIZE)
print(f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}")

Train batches: 5641  |  Test batches: 706


In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    progress = (epoch - 1) / max(EPOCHS - 1, 1)
    alpha = LAMBDA_ADV * (2 / (1 + math.exp(-ADV_ALPHA * progress)) - 1)

    model.train()
    total_tox, total_adv, n_batches = 0.0, 0.0, 0
    for x, y_tox, y_id in train_loader:
        x, y_tox, y_id = x.to(DEVICE), y_tox.to(DEVICE), y_id.to(DEVICE)
        optimizer.zero_grad()
        tox_pred, adv_pred = model(x, alpha=alpha)
        L_tox = F.binary_cross_entropy(tox_pred, y_tox)

        # Apply adversary loss only on rows with at least one identity annotation.
        # Non-annotated rows have y_id=[0,...,0]; including them causes the GRL to push
        # the encoder toward encoding MORE identity information (reversed gradient on
        # all-zero targets), which is the opposite of the debiasing goal.
        has_id = y_id.max(dim=1).values > 0
        if has_id.any():
            L_adv = F.binary_cross_entropy(adv_pred[has_id], y_id[has_id])
            loss  = L_tox + L_adv
        else:
            L_adv = tox_pred.new_tensor(0.0)
            loss  = L_tox

        loss.backward()
        optimizer.step()
        total_tox += L_tox.item()
        total_adv += L_adv.item()
        n_batches += 1
    print(f"Epoch {epoch}/{EPOCHS}  α={alpha:.3f}  tox_loss={total_tox/n_batches:.4f}  adv_loss={total_adv/n_batches:.4f}")

Epoch 1/3  α=0.000  tox_loss=0.1650  adv_loss=0.1529
Epoch 2/3  α=0.231  tox_loss=0.1282  adv_loss=0.1558
Epoch 3/3  α=0.381  tox_loss=0.1205  adv_loss=0.1567


## 5. Inference & Overall Performance

In [7]:
model.eval()
raw_scores: list[float] = []
with torch.no_grad():
    for x, _ in test_loader:
        raw_scores.extend(model(x.to(DEVICE), alpha=0.0)[0].cpu().tolist())

test_df = test_df.copy()
test_df["score"] = raw_scores

overall_auc = roc_auc_score(test_df["toxic"], test_df["score"])
pred_labels = (test_df["score"] >= TOXICITY_THRESHOLD).astype(int)
print(f"Overall AUC          : {overall_auc:.4f}")
print(f"Predicted toxic rate : {pred_labels.mean():.2%}")
print(f"True toxic rate      : {test_df['toxic'].mean():.2%}\n")
print(classification_report(test_df["toxic"], pred_labels, target_names=["non-toxic", "toxic"]))

Overall AUC          : 0.9527
Predicted toxic rate : 5.96%
True toxic rate      : 8.00%

              precision    recall  f1-score   support

   non-toxic       0.96      0.98      0.97    166056
       toxic       0.76      0.57      0.65     14435

    accuracy                           0.95    180491
   macro avg       0.86      0.78      0.81    180491
weighted avg       0.95      0.95      0.95    180491



## 6. Fairness Evaluation

We evaluate fairness using the `ModelBiasEvaluator` from `fairness_jigsaw.metrics`.  
Three families of metrics are computed for each identity subgroup:

| Metric family | What it measures |
|---------------|-----------------|
| **Subgroup AUC** | Discrimination ability within each subgroup only |
| **BPSN AUC** | Background Positive Subgroup Negative — over-flagging non-toxic subgroup comments |
| **BNSP AUC** | Background Negative Subgroup Positive — under-flagging toxic subgroup comments |
| **FPR** | False positive rate — fraction of non-toxic comments incorrectly classified as toxic |
| **ECE** | Expected Calibration Error — how well predicted probabilities match true frequencies |

In [8]:
evaluator = ModelBiasEvaluator(
    identity_cols=DEFAULT_IDENTITY_COLUMNS,
    toxicity_threshold=TOXICITY_THRESHOLD,
    identity_threshold=IDENTITY_THRESHOLD,
    min_subgroup_size=20,
)
results = evaluator.evaluate(test_df, score_col="score", label_col="toxic")

### AUC Metrics

- **Subgroup AUC** close to 1.0 means the model ranks toxic above non-toxic within that demographic.  
- **BPSN AUC** < 0.5 is a red flag: the model is more likely to flag a non-toxic comment mentioning the identity than a toxic background comment.  
- **BNSP AUC** < 0.5 is the dual problem: toxic identity-mentioning comments are ranked below non-toxic background comments.

In [9]:
results["auc"].round(4)

,identity,n,subgroup_auc,bpsn_auc,bnsp_auc,pinned_auc
0,hindu,55,0.7420,0.8928,0.8848,0.8208
1,other_religion,34,0.7517,0.8949,0.9129,0.8329
2,black,1496,0.8036,0.8226,0.9538,0.8464
3,buddhist,49,0.8062,0.8942,0.9247,0.8658
4,white,2551,0.8334,0.8425,0.9569,0.8679
5,homosexual_gay_or_lesbian,1141,0.8339,0.8488,0.9553,0.8704
6,muslim,2151,0.8311,0.8567,0.9504,0.8712
7,transgender,255,0.8438,0.8649,0.9518,0.8800
8,heterosexual,116,0.8382,0.8895,0.9376,0.8829
9,atheist,123,0.8574,0.8932,0.9406,0.8933


### Subgroup FPR

The **False Positive Rate** for each subgroup tells us whether the model disproportionately flags  
non-toxic comments that merely *mention* a demographic group.  
A high FPR for a group (e.g. `black`, `muslim`) relative to the overall FPR indicates systematic over-prediction bias.

In [10]:
results["fpr"].round(4)

,identity,n_negatives,fpr,bg_fpr,fpr_gap
0,black,1039,0.0808,0.0150,0.0659
1,transgender,206,0.0680,0.0153,0.0526
2,white,1839,0.0571,0.0149,0.0422
3,other_race_or_ethnicity,35,0.0571,0.0154,0.0418
4,homosexual_gay_or_lesbian,819,0.0549,0.0152,0.0397
5,latino,166,0.0482,0.0154,0.0328
6,muslim,1662,0.0475,0.0151,0.0325
7,psychiatric_or_mental_illness,365,0.0466,0.0153,0.0313
8,hindu,47,0.0426,0.0154,0.0272
9,atheist,108,0.0370,0.0154,0.0217


### Expected Calibration Error

**ECE** measures how well the model's predicted probabilities match observed toxicity rates within each subgroup.  
A perfectly calibrated model would have ECE = 0 for all groups.  
High ECE for minority groups often indicates the model has seen fewer examples and is less reliable in those contexts.

In [11]:
results["ece"].round(4)

,identity,n,ece
0,other_religion,34,0.1479
1,hindu,55,0.1293
2,heterosexual,116,0.1011
3,latino,209,0.0694
4,transgender,255,0.0692
5,homosexual_gay_or_lesbian,1141,0.0665
6,other_race_or_ethnicity,42,0.0657
7,atheist,123,0.0602
8,black,1496,0.0572
9,buddhist,49,0.0505


## 7. Counterfactual Gap

The counterfactual gap measures how much the model's predicted toxicity score changes when a **single identity word** is swapped for its counterpart (e.g. "black" → "white").  
A large positive gap means the model assigns higher toxicity to the *original* group than the *swapped* group — a form of demographic bias.

> **Note:** This method uses **literal string substitution** and works best for short, unambiguous identity terms.  
> It may miss bias expressed through co-occurrence patterns or multi-word phrases.

In [12]:
def predict_fn(texts: list[str]) -> np.ndarray:
    ids = torch.tensor([encode(t) for t in texts], dtype=torch.long)
    model.eval()
    chunks: list[np.ndarray] = []
    with torch.no_grad():
        for i in range(0, len(ids), BATCH_SIZE):
            chunks.append(model(ids[i:i+BATCH_SIZE].to(DEVICE), alpha=0.0)[0].cpu().numpy())
    return np.concatenate(chunks)

cf_results = evaluator.compute_counterfactual_gap(
    test_df,
    text_col="comment_text",
    score_col="score",
    predict_fn=predict_fn,
    swap_pairs=[("black", "white"), ("christian", "muslim"), ("male", "female")],
)
cf_results.round(4)

,type,term_a,term_b,n_pairs,mean_gap,max_gap
0,neutral,black,person,1917,0.0841,0.8094
1,neutral,muslim,person,1046,0.0734,0.7201
2,neutral,transgender,person,160,0.0623,0.3197
3,neutral,white,person,3875,0.0582,0.7180
4,neutral,christian,person,991,0.0374,0.4953
5,neutral,asian,person,212,0.0223,0.3053
6,neutral,hindu,person,33,0.0186,0.2489
7,neutral,female,person,699,0.0167,0.3632
8,neutral,jewish,person,339,0.0147,0.2140
9,neutral,atheist,person,88,0.0137,0.1452


## Conclusion

**Three-model comparison — averages across 18 subgroups.**

| Metric | Baseline | Reweighting | Adversarial |
|--------|:--------:|:-----------:|:-----------:|
| Overall AUC | 0.9540 | 0.9524 | 0.9527 |
| Predicted toxic rate | 5.30% | 4.80% | 5.96% |
| Toxic recall | 53% | 49% | 57% |
| *Avg subgroup AUC* | 0.847 | 0.837 | **0.851** |
| *Avg BPSN AUC* | 0.876 | **0.904** | 0.880 |
| *Avg BNSP AUC* | **0.946** | 0.926 | **0.946** |
| *Avg pinned AUC* | 0.881 | 0.881 | **0.885** |
| *Avg FPR* | 0.033 | **0.013** | 0.039 |
| *Avg FPR gap* | 0.022 | **0.003** | 0.024 |
| Overall ECE | 0.006 | 0.012 | **0.003** |
| *Avg subgroup ECE* | 0.052 | 0.089 | **0.058** |
| *Avg neutral CF gap* | 0.028 | **0.014** | 0.032 |

**Selected identity metrics (Baseline / Reweighting / Adversarial; bold = best).**

| Identity | BPSN AUC | FPR gap | ECE |
|----------|:--------:|:-------:|:---:|
| `black` | 0.806 / **0.874** / 0.823 | +0.059 / **+0.009** / +0.066 | 0.059 / 0.161 / **0.057** |
| `homosexual_gay_or_lesbian` | 0.806 / **0.863** / 0.849 | +0.051 / **+0.008** / +0.040 | **0.038** / 0.133 / 0.067 |
| `transgender` | 0.845 / **0.878** / 0.865 | +0.051 / **+0.020** / +0.053 | **0.062** / 0.091 / 0.069 |
| `muslim` | 0.840 / **0.885** / 0.857 | +0.022 / **−0.002** / +0.033 | **0.034** / 0.105 / 0.036 |
| `white` | 0.823 / **0.886** / 0.843 | +0.034 / **+0.003** / +0.042 | 0.051 / 0.143 / **0.042** |
| `christian` | 0.932 / **0.934** / 0.906 | +0.003 / **−0.003** / +0.006 | **0.015** / 0.026 / 0.022 |

---

**Overall performance.** AUC = 0.9527 (≈ baseline). Predicted toxic rate rises to 5.96% vs. reweighting's 4.80%, indicating less conservative scoring; recall improves to 57% (vs. 49%).

**AUC.** Avg pinned AUC reaches 0.885, marginally best of the three. Worst subgroup AUC: `hindu` (0.742) and `other_religion` (0.752), both small-n. Avg BPSN AUC declines vs. reweighting (0.904 → 0.880): the model over-ranks non-toxic identity-mentioning comments relative to toxic background comments more than reweighting did, notably for `black` (0.874 → 0.823) and `white` (0.886 → 0.843).

**FPR.** Avg FPR (0.039) and avg FPR gap (0.024) both exceed the unmitigated baseline. `black` FPR gap is the worst at +0.066 (7× the reweighting value); `transgender` at +0.053. The higher predicted toxic rate means more comments cross the 0.5 threshold, amplifying any residual identity–score correlation.

**ECE.** Overall ECE 0.003 (best of three). Avg subgroup ECE (0.058) is much better than reweighting (0.089) and close to baseline (0.052). Large improvements for `black` (0.161 → 0.057), `white` (0.143 → 0.042), and `homosexual_gay_or_lesbian` (0.133 → 0.067).

**Counterfactual gap.** Avg neutral CF gap (0.032) worsens vs. both baseline (0.028) and reweighting (0.014). `black` → `person`: 0.084 (2.6× reweighting); `muslim` → `person`: 0.073 (2.6×). The adversary loss barely moved during training (0.153 → 0.157): `LAMBDA_ADV = 0.5` over 3 epochs exerted insufficient pressure to remove identity-correlating signals from the encoder.

**Takeaway.** Neither mitigation dominates across all metrics: reweighting wins on FPR and CF gap; adversarial debiasing wins on calibration. To improve further, consider increasing `LAMBDA_ADV`, extending training, or stacking adversarial debiasing on top of reweighting.